# Chunking Strategies: simple vs. semantic vs. hybrid

## Why chunking matters

Whole documents are a poor unit for retrieval. An embedding model has an input limit, and one vector for a long document averages over every topic in it. An LLM has a finite context window, and each irrelevant passage passed to it costs tokens and can distract it. So ingestion pipelines split extracted text into **chunks**, embed each chunk, and store the vectors in an index. At query time the query is embedded, the nearest chunks are returned, and they are passed to an LLM (retrieval-augmented generation, RAG) or shown to a user (search).

The chunk is therefore the unit of retrieval, and its boundaries decide what a retriever can return. A boundary through a table row, or a chunk that mixes two topics, lowers retrieval quality regardless of the embedding model.

In docling-pipelines the chunker sits between extraction and embedding:

```text
ingest_source --> extract_operator --> chunker --> embeddings --> vectordb
```

This notebook covers the `chunker` step. It stops after chunking and uses its own embedding and search code in place of the `embeddings` and `vectordb` operators, so the strategies can be compared directly. Those two operators are covered in notebooks `04_embeddings_vectordb` and `06_rag_pipeline`.

## What this notebook compares

`ChunkerOperator` supports three strategies. This notebook runs all three on the same documents in a single flow, compares the resulting chunks structurally and by embedding coherence, and tests which strategy retrieves the right passage for a small set of queries.

## The three strategies

| Strategy | Boundary rule | Strength | Limitation |
|---|---|---|---|
| `simple` | Splits on `.`, paragraph, newline or space separators until pieces fit within `chunk_size` characters; tries to carry up to `chunk_overlap` characters into the next chunk | Deterministic, fast, no dependencies | Layout-blind; `chunk_size` is an upper bound and overlap can be zero when a separator boundary falls before the overlap window |
| `hybrid` | Docling's `HybridChunker`: walks the parsed document tree (headings, paragraphs, tables, lists) and packs nodes into chunks of at most `chunk_size` tokens | Keeps tables, lists and sections intact | Gains depend on the input having real structure |
| `semantic` | LangChain `SemanticChunker`: splits into sentences, embeds them, and starts a new chunk where the cosine distance between neighbouring sentences exceeds a percentile threshold | Follows topic shifts instead of layout | Requires an embedding model; chunk sizes are unbounded; it can only split at sentence punctuation, so code blocks and tables are not split finely |

For `semantic`, `breakpoint_threshold_amount` is the percentile of all neighbour distances in a document above which a boundary is placed. At 95 only the largest 5% of jumps become boundaries; at 45 the largest 55% do. The threshold is relative to each document, so a lower value always gives more, smaller chunks.

Parameter reference: `docs/operators/functional/chunker_readme.md` in the repository.

## What the notebook does

| Step | What it does | Runs where |
|---|---|---|
| 1. Ingest | Reads three markdown files from this repository's `docs/` folder | Local |
| 2. Extract | Docling parses each file into structured text | Local |
| 3. Chunk | A `branching` node fans the extract output out to three chunker nodes | Local; `semantic` calls Ollama |
| 4. Compare | Chunk counts and sizes, then embedding-based coherence | Local + Ollama |
| 5. Retrieve | Embeds five queries and checks which strategy returns the right chunk | Local + Ollama |

Parameter values and queries are illustrative. Results change with the documents, the settings and the embedding model, so compare strategies with each other rather than reading a single number as a verdict.

## Prerequisites

- Virtual environment activated and `PYTHONPATH` set (see `examples/notebooks/README.md`)
- **Ollama is required**: running at `http://localhost:11434` with `nomic-embed-text` pulled. The notebook checks this early and stops with instructions if it is missing.

## 1. Setup

Imports first. Nothing runs a pipeline yet.

In [ ]:
import html
import os
import sys
from itertools import pairwise
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import requests
from IPython.display import HTML, display

# Add src to path if needed
if "PYTHONPATH" not in os.environ:
    sys.path.insert(0, str(Path.cwd().parent.parent / "src"))

from docpipe.lib.docpipe_flow_manager import DocpipeFlowManager
from docpipe.utils.infrastructure.filesystem import get_data_path

print("Imports loaded successfully")

All tunable settings are in the next cell. After changing the documents or a chunker setting, re-run from section 3 onward.

`SEMANTIC_PERCENTILE` is set to 45 rather than the library default of 95. At 95 the semantic chunker produced a few very large chunks on these documents, so the structural comparison would have been dominated by size. 45 brings its mean chunk size into the same range as the other two strategies. It was picked from a sweep of chunk sizes, not from retrieval results, so the setting is not tuned to the queries below.

In [ ]:
# Find the repository root by walking up from the working directory, so this works from any folder inside the repo.
REPO_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs" / "guides" / "FLOW_CONFIGURATION_GUIDE.md").exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the docling-pipelines repository")

# Documents to chunk (relative to the repository root)
CORPUS = [
    "docs/guides/FLOW_CONFIGURATION_GUIDE.md",
    "docs/guides/PYTHON_API_GUIDE.md",
    "docs/guides/SECURITY_BEST_PRACTICES.md",
]

# Ollama and the embedding model used for semantic chunking, coherence scores and retrieval
OLLAMA_HOST = "http://localhost:11434"
EMBEDDING_MODEL = "nomic-embed-text"

# Chunker settings
SIMPLE_CHUNK_SIZE = 1000  # characters
SIMPLE_CHUNK_OVERLAP = 200  # characters
HYBRID_CHUNK_SIZE = 512  # tokens
HYBRID_CHUNK_OVERLAP = 50
HYBRID_TOKENIZER = "sentence-transformers/all-MiniLM-L6-v2"
SEMANTIC_PERCENTILE = 45.0  # lower = more, smaller chunks

# The flow writes its output under data/<JOB_ID>/<JOB_RUN_ID>/. Fixed IDs make that path predictable.
JOB_ID = "chunking-strategies-nb"
JOB_RUN_ID = "run"

# Retrieval comparison
TOP_K = 3  # chunks returned per strategy
MAX_EMBED_CHARS = 4000  # nomic-embed-text rejects long inputs, so only this much of each text is embedded
MAX_SENTENCES_PER_CHUNK = 20  # bounds the embedding calls used for coherence scores

# Each query has an "answer phrase": text the correct chunk must contain.
# A strategy succeeds on a query if a chunk containing the phrase is among its top results.
QUERIES = [
    {
        "label": "Secret rotation (table row)",
        "query": "How often should the JWT secret key be rotated?",
        "answer_phrase": "immediately after any suspected exposure",
        "where": "SECURITY_BEST_PRACTICES.md, one row of a table",
    },
    {
        "label": "job_id generation (list)",
        "query": "How is a job_id generated from the flow_name?",
        "answer_phrase": "UUID v5",
        "where": "FLOW_CONFIGURATION_GUIDE.md, a bulleted and numbered list",
    },
    {
        "label": "No job_id given (one sentence)",
        "query": "What happens if I do not pass a job_id to DocpipeFlowManager?",
        "answer_phrase": "random UUID will be generated",
        "where": "FLOW_CONFIGURATION_GUIDE.md, a single sentence under a heading",
    },
    {
        "label": "Embedded logging (code block)",
        "query": "How do I stop Docling Pipelines from setting up logging when I use it inside my own application?",
        "answer_phrase": "configure_logging=False",
        "where": "PYTHON_API_GUIDE.md, a short paragraph plus a code block",
    },
    {
        "label": "Control (no answer exists)",
        "query": "What is the difference between in-memory and filesystem storage?",
        "answer_phrase": None,
        "where": "nowhere: the documents only use the word 'in-memory' in JSON examples",
    },
]

STRATEGIES = ["simple", "hybrid", "semantic"]
DOC_NAMES = [Path(p).name for p in CORPUS]

print(f"Repository root: {REPO_ROOT}")
print("Documents:")
for path in CORPUS:
    print(f"  - {path}")

### Ollama is required

Semantic chunking, the coherence scores and the retrieval comparison all call Ollama, and there is no fallback path. If Ollama or the model is missing, the next cell prints the command to run and raises, so "Run All" stops here instead of producing partial results.

In [ ]:
def ollama_problem():
    """Return a description of what is missing, or None if Ollama and the model are ready."""
    try:
        response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=3)
        response.raise_for_status()
    except requests.exceptions.RequestException as exc:
        return f"Ollama is not reachable at {OLLAMA_HOST} ({type(exc).__name__}).\nStart it with:  ollama serve"

    models = [m["name"] for m in response.json().get("models", [])]
    if not any(EMBEDDING_MODEL in name for name in models):
        return f"Ollama is running, but '{EMBEDDING_MODEL}' is not installed.\nInstall it with:  ollama pull {EMBEDDING_MODEL}"
    return None


problem = ollama_problem()
if problem:
    raise RuntimeError(f"{problem}\nThen re-run this notebook.")
print(f"Ollama is reachable and '{EMBEDDING_MODEL}' is installed")

### Helper functions

Embedding via Ollama (with a per-text cache), cosine similarity, and sentence splitting. Run the next cell, since sections 6 and 7 call these functions. It only defines them and prints nothing; read it only if you want to see how the scores are computed.

In [ ]:
EMBED_CACHE = {}


def embed_texts(texts, batch_size=32):
    """Embed texts with Ollama, caching results so each text is embedded only once."""
    pending = [t for t in dict.fromkeys(texts) if t not in EMBED_CACHE]
    for start in range(0, len(pending), batch_size):
        batch = pending[start : start + batch_size]
        trimmed = [t[:MAX_EMBED_CHARS] for t in batch]
        response = requests.post(
            f"{OLLAMA_HOST}/api/embed",
            json={"model": EMBEDDING_MODEL, "input": trimmed},
            timeout=300,
        )
        response.raise_for_status()
        for text, vector in zip(batch, response.json()["embeddings"], strict=True):
            EMBED_CACHE[text] = np.asarray(vector, dtype=float)
    return np.vstack([EMBED_CACHE[t] for t in texts])


def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def mean_pairwise_cosine(vectors):
    n = len(vectors)
    if n < 2:
        return float("nan")
    normed = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
    sims = normed @ normed.T
    return float((sims.sum() - n) / (n * (n - 1)))


def split_sentences(text):
    sentences = [s.strip() for s in text.split(". ") if len(s.strip()) > 20]
    return sentences[:MAX_SENTENCES_PER_CHUNK]


def squash(text):
    """Collapse whitespace so phrase matching ignores line breaks."""
    return " ".join(text.split())

## 2. The documents

Three files from this repository's own documentation, picked for their different content shapes:

| File | Content | What it exercises |
|---|---|---|
| `FLOW_CONFIGURATION_GUIDE.md` | Long explanations, bulleted and numbered lists, many JSON examples | Lists and prose with topic shifts |
| `PYTHON_API_GUIDE.md` | Short how-to sections, each followed by a code block | Code blocks with almost no sentence punctuation |
| `SECURITY_BEST_PRACTICES.md` | Many short headed sections and several small tables | Table rows that only make sense next to their header |

## 3. Build and run one flow

A single trunk fans out into three chunker nodes through a `branching` node:

```text
ingest --> extract --> split --+--> chunk_simple
                               |
                               +--> chunk_hybrid
                               |
                               +--> chunk_semantic
```

`extract` runs once, and the `split` node (a `branching` operator) copies its whole output to each of its three branches. Branches with no criteria are unconditional, so every chunker receives every document and every strategy sees identical input text. In the flow JSON each branch is a key under `branches`, and a downstream node names it as `<branching node>.<branch>` in `depends_on`. With `data_storage_type: "local"` each node writes parquet under `data/<JOB_ID>/<JOB_RUN_ID>/data/<node name>_<n>/`, where `<n>` numbers the node's own output tables. A chunker has one output, so it is always `_0`; the `split` node has three, so it writes `split_0` to `split_2`. Micro-batching is on by default and adds a batch-number folder below that, for example `chunk_simple_0/0/output.parquet`.

The cell takes a minute or two (Docling loads its models and the semantic chunker calls Ollama). Every run executes the whole flow and overwrites the previous output under the same `JOB_ID`. `force_ingest: true` in `global_config` is what makes repeat runs safe: the ingest step tracks which files it has already processed per `job_id`, and without this flag a second run would see the three documents as already done and pass nothing downstream (the old output files would stay in place, so the notebook would silently read stale chunks).

Notes on the flow design and the output you will see:

- **No overwriting between branches.** Each branch gets its own copy of the table, and each chunker adds its own `chunked_content` column to that copy and writes it to its own folder (`chunk_simple_0`, `chunk_hybrid_0`, `chunk_semantic_0`). The three results never share a table or a file. The notebook does not read the `split_*` folders.
- **Validation warnings.** The flow validator prints a `GENERATE_OUTPUT_MISSING` warning for each chunker node: *The last operator should be a "Vector DB" or "Storage" operator*. It is expected. A production flow ends in a `vectordb` or `document_set` node, whereas this flow stops at the chunkers so the notebook can read their output directly. The warning does not affect the results.
- **`transformers` warning.** You may see *"Token indices sequence length is longer than the specified maximum sequence length (514 > 512)"*. The hybrid chunker's tokenizer is only counting tokens for chunk sizing; nothing is truncated, so the warning is harmless.

In [ ]:
CHUNK_NODES = {
    "simple": "chunk_simple",
    "hybrid": "chunk_hybrid",
    "semantic": "chunk_semantic",
}
RUN_DATA_DIR = Path(get_data_path()) / JOB_ID / JOB_RUN_ID / "data"


corpus_paths = [str(REPO_ROOT / p) for p in CORPUS]
missing = [p for p in corpus_paths if not Path(p).exists()]
if missing:
    raise FileNotFoundError(f"Documents not found: {missing}")

chunker_configs = {
    "chunk_simple": {
        "chunk_type": "simple",
        "chunk_size": SIMPLE_CHUNK_SIZE,
        "chunk_overlap": SIMPLE_CHUNK_OVERLAP,
    },
    "chunk_hybrid": {
        "chunk_type": "hybrid",
        "chunk_size": HYBRID_CHUNK_SIZE,
        "chunk_overlap": HYBRID_CHUNK_OVERLAP,
        "docling_tokenizer": HYBRID_TOKENIZER,
    },
    "chunk_semantic": {
        "chunk_type": "semantic",
        "semantic_embeddings_model": EMBEDDING_MODEL,
        "breakpoint_threshold_type": "percentile",
        "breakpoint_threshold_amount": SEMANTIC_PERCENTILE,
        "provider_config": {"host": OLLAMA_HOST},
    },
}

flow = {
    "flow_name": "chunking-strategies-comparison",
    "description": "Ingest and extract three documents, fan out, then chunk them three ways",
    "global_config": {
        "doc_column": "content",
        "disable_validation": False,
        "force_ingest": True,
        "data_storage_type": "local",  # write parquet so the chunks can be read back
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "provider_config": {"paths": corpus_paths},
                "include_filter": "md",
            },
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {"provider": "docling_library", "doc_column": "content"},
                "entity_extraction": {"provider": "none"},
            },
        },
        {
            "name": "split",
            "type": "branching",
            "depends_on": ["extract"],
            # No criteria: each branch receives the whole table (fan-out).
            "config": {"branches": {f"to_{strategy}": {} for strategy in STRATEGIES}},
        },
        *[
            {
                "name": node,
                "type": "chunker",
                "depends_on": [f"split.to_{strategy}"],
                "config": {**chunker_configs[node], "doc_column": "content"},
            }
            for strategy, node in CHUNK_NODES.items()
        ],
    ],
}

manager = DocpipeFlowManager(flow_def=flow, job_id=JOB_ID, job_run_id=JOB_RUN_ID)
manager.execute()
print(f"Flow finished; output is in {RUN_DATA_DIR}")

Read the chunker outputs back. Each output row is one document; its `chunked_content` column holds the list of chunks, each with `chunk`, `chunk_sequence_number` and `start_index`.

In [ ]:
def load_chunks(node):
    """Return {document name: [chunk text, ...]} for one chunker node."""
    chunks = {}
    for path in sorted((RUN_DATA_DIR / f"{node}_0").rglob("output.parquet")):
        for row in pq.read_table(path).to_pylist():
            ordered = sorted(row["chunked_content"], key=lambda c: c["chunk_sequence_number"])
            chunks[row["name"]] = [c["chunk"] for c in ordered]
    return chunks


CHUNKS = {strategy: load_chunks(node) for strategy, node in CHUNK_NODES.items()}

for strategy in STRATEGIES:
    absent = [name for name in DOC_NAMES if not CHUNKS[strategy].get(name)]
    if absent:
        raise RuntimeError(f"'{strategy}' produced no chunks for: {absent}")

counts = pd.DataFrame(
    {strategy: {name: len(CHUNKS[strategy][name]) for name in DOC_NAMES} for strategy in STRATEGIES}
)
counts.index.name = "chunks per document"
display(counts)

**Points to notice**

- Every strategy produced chunks for every document. This is checked explicitly because the semantic chunker fails when a single "sentence" (a long run with no sentence punctuation, such as a large table) exceeds the embedding model's context window.
- Different counts reflect different boundary rules, not correctness. A higher count is not better on its own; the next sections test whether the boundaries are useful.

## 4. Inspect the chunks

Read some chunks before looking at aggregates. `SECURITY_BEST_PRACTICES.md` has a rotation table with one row per credential. The next cell first shows the original markdown section that holds the `JWT_SECRET_KEY` row, then the chunk containing that row from each strategy, one below the other. The question is whether the row stays attached to the context that gives it meaning.

The source section comes from the file on disk. The chunkers read Docling's extracted text, which may differ slightly in whitespace and table formatting.

In [ ]:
def chunk_with_phrase(strategy, doc_name, phrase):
    for index, text in enumerate(CHUNKS[strategy][doc_name]):
        if phrase in squash(text):
            return index, text
    return None, None


def source_section(doc_name, phrase):
    """Return the markdown section (heading to next heading) of the source file that contains `phrase`."""
    path = next(REPO_ROOT / p for p in CORPUS if Path(p).name == doc_name)
    lines = path.read_text().splitlines()
    hit = next((i for i, line in enumerate(lines) if phrase in line), None)
    if hit is None:
        return None
    start = next((i for i in range(hit, -1, -1) if lines[i].startswith("#")), 0)
    end = next((i for i in range(hit + 1, len(lines)) if lines[i].startswith("#")), len(lines))
    return "\n".join(lines[start:end]).strip()


def show_chunks(doc_name, phrase, limit=900):
    """Show the source section for `phrase`, then the chunk containing it from each strategy."""
    section = source_section(doc_name, phrase)
    blocks = []
    if section is not None:
        blocks.append(
            "<div style='border:2px solid #555;padding:6px 10px;margin:8px 0'>"
            f"<b>original: {html.escape(doc_name)}, {len(section)} characters</b>"
            f"<pre style='white-space:pre-wrap;margin:6px 0 0'>{html.escape(section)}</pre></div>"
        )
    for strategy in STRATEGIES:
        index, text = chunk_with_phrase(strategy, doc_name, phrase)
        if text is None:
            title = f"{strategy}: no chunk contains this phrase"
            body = ""
        else:
            title = f"{strategy}: chunk #{index}, {len(text)} characters"
            body = html.escape(text[:limit]) + (" ..." if len(text) > limit else "")
        blocks.append(
            "<div style='border:1px solid #999;padding:6px 10px;margin:8px 0'>"
            f"<b>{title}</b><pre style='white-space:pre-wrap;margin:6px 0 0'>{body}</pre></div>"
        )
    display(HTML("".join(blocks)))


show_chunks("SECURITY_BEST_PRACTICES.md", "immediately after any suspected exposure")

**Points to notice**

- Compare each chunk with the original section. Is the row still accompanied by its column header or section heading, or does it appear as a bare line?
- Is the chunk focused on rotation, or does it also contain unrelated sections? Mixed chunks dilute the embedding.
- How does each strategy represent the table? `hybrid` serialises table cells itself, so check whether the format differs from the raw markdown that `simple` and `semantic` see.
- Compare chunk lengths. `simple` is capped in characters, `hybrid` in tokens, and `semantic` has no cap.

## 5. Structural metrics

One table, grouped by document. Within each group the three rows are the strategies run on the same text, so read down a column to compare them.

| Column | Meaning |
|---|---|
| `chunks` | Number of chunks |
| `mean_chars`, `std_chars` | Mean and standard deviation of chunk length in characters |
| `min_chars`, `max_chars` | Shortest and longest chunk |
| `overlap_ratio` | Mean share of each chunk's characters that exactly repeat the end of the previous chunk |

In [ ]:
def overlap_chars(prev, nxt, limit=500):
    for k in range(min(limit, len(prev), len(nxt)), 0, -1):
        if prev.endswith(nxt[:k]):
            return k
    return 0


rows = []
for strategy in STRATEGIES:
    for doc_name in DOC_NAMES:
        texts = CHUNKS[strategy][doc_name]
        sizes = np.array([len(t) for t in texts])
        ratios = [overlap_chars(a, b) / len(b) for a, b in pairwise(texts) if b]
        rows.append(
            {
                "document": doc_name,
                "strategy": strategy,
                "chunks": len(texts),
                "mean_chars": round(float(sizes.mean())),
                "std_chars": round(float(sizes.std())),
                "min_chars": int(sizes.min()),
                "max_chars": int(sizes.max()),
                "overlap_ratio": round(float(np.mean(ratios)), 3) if ratios else 0.0,
            }
        )

STRUCTURE = pd.DataFrame(rows).set_index(["document", "strategy"]).sort_index()
display(STRUCTURE)

**Points to notice**

- `simple` uses `SIMPLE_CHUNK_SIZE` as an upper bound. It splits on separators (`.`, `\n\n`, `\n`, space), so chunks can be shorter and `overlap_ratio` may be zero when a separator boundary lands before the overlap window would start.
- `hybrid` splits on document structure and does not repeat text, so `overlap_ratio` should be close to 0.
- Compare `max_chars` across strategies. A chunk far larger than the rest is mostly an averaged embedding of several topics. The semantic chunker splits only at sentence punctuation, so a code block or table without it can stay whole even at a low percentile.
- `mean_chars` for `semantic` follows `SEMANTIC_PERCENTILE`. Change it, re-run from section 3, and watch the mean move.

## 6. Embedding coherence

Chunk size says little about whether boundaries are good. Two cosine-similarity measures, computed with `nomic-embed-text`, give a proxy:

- **`intra_chunk_sim`** (higher is better): mean pairwise cosine similarity between the sentence embeddings inside a chunk. High means the chunk is about one thing.
- **`inter_chunk_sim`** (lower means more distinct): cosine similarity between the mean sentence embeddings of adjacent chunks. Very high means the boundary fell inside a topic.

Embeddings map text to vectors so that passages with similar meaning end up close together, and cosine similarity measures that closeness. Applied to chunking, it gives a proxy for boundary quality: sentences inside a good chunk should be close to each other, and neighbouring chunks should be further apart than the sentences within them. It does not prove the chunks are good, since it depends on the embedding model. It is cheap to compute, though, and it is the same signal the semantic chunker uses to place its boundaries.

Simplifications that affect how to read the numbers:

- Sentences are split on `". "` rather than with a linguistic sentence tokenizer, because the goal is to compare strategies, not to benchmark sentence splitting.
- Absolute values depend on the vocabulary of each document, so compare strategies within one document.
- Only the first `MAX_EMBED_CHARS` characters of any text are embedded.

Embedding is the slow step: each batch is a request to Ollama. The next cell therefore embeds every chunk once and keeps the vectors. They are reused in section 7 as the search index, so no chunk is embedded twice.

In [ ]:
CHUNK_RECORDS = [
    {"strategy": strategy, "document": doc_name, "index": index, "text": text}
    for strategy in STRATEGIES
    for doc_name in DOC_NAMES
    for index, text in enumerate(CHUNKS[strategy][doc_name])
]
CHUNK_VECS = embed_texts([r["text"] for r in CHUNK_RECORDS])
print(f"Embedded {len(CHUNK_RECORDS)} chunks; embedding array shape: {CHUNK_VECS.shape}")

In [ ]:
sentence_lists = [split_sentences(r["text"]) for r in CHUNK_RECORDS]
embed_texts([s for sentences in sentence_lists for s in sentences])  # one pass, results cached

for record, sentences, chunk_vec in zip(CHUNK_RECORDS, sentence_lists, CHUNK_VECS, strict=True):
    if sentences:
        vectors = np.vstack([EMBED_CACHE[s] for s in sentences])
        record["intra"] = mean_pairwise_cosine(vectors)
        record["mean_vec"] = vectors.mean(axis=0)
    else:
        record["intra"] = float("nan")
        record["mean_vec"] = chunk_vec

rows = []
for strategy in STRATEGIES:
    for doc_name in DOC_NAMES:
        group = [r for r in CHUNK_RECORDS if r["strategy"] == strategy and r["document"] == doc_name]
        inter = [cosine(a["mean_vec"], b["mean_vec"]) for a, b in pairwise(group)]
        intra = [r["intra"] for r in group if not np.isnan(r["intra"])]
        rows.append(
            {
                "document": doc_name,
                "strategy": strategy,
                "chunks": len(group),
                "intra_chunk_sim": round(float(np.mean(intra)), 3) if intra else float("nan"),
                "inter_chunk_sim": round(float(np.mean(inter)), 3) if inter else float("nan"),
            }
        )

COHERENCE = pd.DataFrame(rows).set_index(["document", "strategy"]).sort_index()
display(COHERENCE)

**Points to notice**

- The pattern of good boundaries is higher `intra_chunk_sim` together with lower `inter_chunk_sim` on the same document.
- Compare strategies within a document, not across documents.
- Chunks with fewer than two usable sentences (for example a lone heading) have no intra score and are excluded from the mean.
- Coherence is a proxy. The next section measures the outcome that matters: whether retrieval finds the answer.

## 7. Retrieval

This mimics the retrieval step of a RAG system: embed the query, rank chunks by cosine similarity, return the top `TOP_K`.

There are five queries. Four have an answer in the documents, each in a different content shape (table row, list, single sentence, code block). The fifth is a **control** with no answer in the corpus; its scores show what a non-match looks like and give a baseline for reading the others.

Each answerable query has an **answer phrase**, a substring that the correct chunk must contain. A strategy succeeds on a query if a chunk containing the phrase is in its top `TOP_K`, and the summary reports the rank of the first such chunk. This is a substring check, not a labelled dataset. The chunk text is printed after the summary so you can judge quality directly.

Each summary cell also shows the **top-1 similarity**: the cosine similarity between the query and the highest-ranked chunk for that strategy, whether or not that chunk contains the answer phrase. It is the raw retrieval score, so it shows how confident the retriever was, not whether it was right.

In [ ]:
NORMED_VECS = CHUNK_VECS / np.linalg.norm(CHUNK_VECS, axis=1, keepdims=True)
STRATEGY_IDX = {s: [i for i, r in enumerate(CHUNK_RECORDS) if r["strategy"] == s] for s in STRATEGIES}


def search(query, strategy):
    """Return the TOP_K (score, record) pairs for one strategy, best first."""
    query_vec = embed_texts([query])[0]
    query_vec = query_vec / np.linalg.norm(query_vec)
    idx = STRATEGY_IDX[strategy]
    scores = NORMED_VECS[idx] @ query_vec
    order = np.argsort(scores)[::-1][:TOP_K]
    return [(float(scores[i]), CHUNK_RECORDS[idx[i]]) for i in order]


def first_hit_rank(hits, phrase):
    for rank, (_, record) in enumerate(hits, start=1):
        if phrase in squash(record["text"]):
            return rank
    return None


RESULTS = {}  # (query label, strategy) -> list of (score, record)
summary_rows = []
for spec in QUERIES:
    row = {"query": spec["label"]}
    for strategy in STRATEGIES:
        hits = search(spec["query"], strategy)
        RESULTS[(spec["label"], strategy)] = hits
        best = hits[0][0]
        if spec["answer_phrase"] is None:
            row[strategy] = f"no answer exists | top-1 similarity {best:.2f}"
        else:
            rank = first_hit_rank(hits, spec["answer_phrase"])
            found = f"found at rank {rank}" if rank else f"not in top {TOP_K}"
            row[strategy] = f"{found} | top-1 similarity {best:.2f}"
    summary_rows.append(row)

SUMMARY = pd.DataFrame(summary_rows).set_index("query")
display(SUMMARY.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))

**Points to notice**

- Rank 1 means the first chunk returned contained the answer phrase. Rank 3 still counts as found, but an LLM consuming the results would read two irrelevant chunks first.
- A high top-1 similarity does not imply the right chunk. If the control query scores close to the real queries, similarity scores alone cannot separate good matches from bad ones.
- Rows can disagree: a strategy that handles the table query may handle the code-block query worse. That is why the queries cover different content shapes.
- Five queries on three documents is a demonstration of the method, not a benchmark. Repeat it on your own documents and queries before choosing.

### Returned chunks

The top chunks per strategy for each query, with score and excerpt. `*` marks a chunk containing the answer phrase.

In [ ]:
def excerpt(text, width=260):
    flat = squash(text)
    return flat[:width] + (" ..." if len(flat) > width else "")


for spec in QUERIES:
    print(f"\nQuery: {spec['query']}")
    print(f"  answer location: {spec['where']}")
    columns = {}
    for strategy in STRATEGIES:
        cells_out = []
        for score, record in RESULTS[(spec["label"], strategy)]:
            hit = spec["answer_phrase"] is not None and spec["answer_phrase"] in squash(record["text"])
            marker = "* " if hit else ""
            cells_out.append(
                f"{marker}[{record['document']} #{record['index']}] score {score:.2f}\n{excerpt(record['text'])}"
            )
        columns[strategy] = cells_out
    detail = pd.DataFrame(columns, index=[f"top {i + 1}" for i in range(TOP_K)])
    display(
        detail.style.set_properties(**{"white-space": "pre-wrap", "text-align": "left", "vertical-align": "top"})
    )

**Points to notice**

- A `*` shows the phrase is present, not that the chunk is a good answer. Check whether the passage is focused or the phrase is buried in a long unrelated block.
- For the control query, check that nothing returned actually answers it.
- A result that starts mid-sentence comes from the character-window cut in `simple`.

## Reading the results

The observations below come from one run with the settings in this notebook (`nomic-embed-text`, `SEMANTIC_PERCENTILE = 45`). Your tables will differ in detail. Treat the direction of each difference as the finding and ignore the exact digits. With five queries on three documents the differences are small, so this section describes what to look for, not a winner.

**All three strategies found every answer.** In our run each strategy had a chunk containing the answer phrase in its top 3 for all four answerable queries, and the ranks were similar (a mix of rank 1 and 2, with one rank 3 for `simple`). What differed was the shape of the chunk that carried the answer, which matters more once an LLM has to read it.

**Tables and lists.** For the rotation table, `hybrid` returned one 589-character chunk holding the whole table, with each row rewritten as `column, Header = value`, so a value stays attached to its column name even without the header row. `simple` also kept the header and several rows together, but a fixed window can end partway through a table or start with a leftover `.` from the previous cut. `semantic` returned a chunk that combined an unrelated JSON example, the "Rotate secrets regularly" heading and the table, and a different, related chunk about the JWT secret ranked above it. For the `job_id` list, `simple` and `semantic` ranked the answer first and `hybrid` ranked it second, behind a heading-only chunk of about 160 characters. Short heading chunks are a side effect of splitting on structure.

**Short answers under a heading.** The "random UUID" sentence is a single line. `simple` placed it third, behind two chunks about `job_id` generation that are related but answer a different question. `hybrid` and `semantic` placed it first. A fixed window that lands on a related passage can outrank the passage that holds the answer.

**Code blocks.** The logging query was found at rank 1 by all three, and `semantic` scored highest because it isolated the short heading-plus-paragraph as its own small chunk. Code blocks are also where `semantic` is weakest: they contain few sentence-ending punctuation marks, so its largest chunks stayed in the thousands of characters at every percentile we tried, and lowering `SEMANTIC_PERCENTILE` further does not split them because the splitter can only cut between "sentences". It also produced a few very small fragments, down to 5 characters.

**Coherence.** `hybrid` had the highest `intra_chunk_sim` on two of the three documents and was effectively tied with `simple` on `PYTHON_API_GUIDE.md`. `semantic` was below `simple` on two of three. `simple` had the highest `inter_chunk_sim` on every document, which is probably at least partly mechanical: the `simple` strategy carries up to `SIMPLE_CHUNK_OVERLAP` characters into the next chunk when a separator boundary permits, so neighbouring chunks can look alike. Do not read it as `simple` choosing worse boundaries. The gaps between strategies were small (hundredths on `intra_chunk_sim`, up to about a tenth on `inter_chunk_sim`), so a different embedding model could change the ordering.

**The control query.** The no-answer query scored lower than the answerable ones, but the margin depends on the strategy. For `semantic`, the control query's top-1 similarity was close to its top-1 similarity on the rotation query, so a raw similarity alone would not reliably separate a real match from a non-match there. This is the practical reason to check retrieval with an answer phrase, and to calibrate any score threshold against a control like this one on your own data.

**What the run suggests.** On this small test, retrieval success was close to a tie, so the choice comes down to what you want the chunks to look like. `hybrid` kept the rotation table in one chunk and attached table values to their column names, at the cost of a few tiny heading-only chunks. `simple` was a reasonable baseline whose weak spot is a boundary that lands on a related but wrong passage. `semantic` can isolate short, focused passages in prose, but it needs a tuned threshold, produces very uneven sizes, and cannot split text that has no sentence punctuation. Repeat the exercise on your own documents and queries before choosing.

## 8. Choosing a strategy

| Input | Start with | Reason |
|---|---|---|
| Structured documents (headings, tables, lists, code) | `hybrid` | Packs Docling's structural nodes, so a table row stays with its header |
| Flat text with no useful structure (notes, transcripts) | `simple` | Cheap and deterministic; overlap guards against a sentence cut at the boundary |
| Long prose that shifts topic without headings, where embedding calls are affordable | `semantic`, then tune `breakpoint_threshold_amount` | Follows meaning, but check chunk sizes first; the default of 95 can yield very large chunks |

Guidelines:

- Choose the cheapest strategy that keeps one answer inside one chunk.
- `chunk_size` is in characters for `simple` and tokens for `hybrid`; `semantic` ignores it.
- Keep chunks within the embedding model's input limit; longer chunks are only partly embedded.
- Validate the choice on your own queries with answer phrases, as in section 7.

### Config quick reference

`simple`:

```json
{"type": "chunker", "name": "chunk", "config": {"chunk_type": "simple", "chunk_size": 1000, "chunk_overlap": 200}, "depends_on": ["extract"]}
```

`hybrid`:

```json
{"type": "chunker", "name": "chunk", "config": {"chunk_type": "hybrid", "chunk_size": 512, "chunk_overlap": 50, "docling_tokenizer": "sentence-transformers/all-MiniLM-L6-v2"}, "depends_on": ["extract"]}
```

`semantic` (requires Ollama):

```json
{"type": "chunker", "name": "chunk", "config": {"chunk_type": "semantic", "semantic_embeddings_model": "nomic-embed-text", "breakpoint_threshold_type": "percentile", "breakpoint_threshold_amount": 45.0}, "depends_on": ["extract"]}
```

Full parameter reference: `docs/operators/functional/chunker_readme.md` in the repository.

### Cleanup

The flow writes its output to `data/chunking-strategies-nb/` relative to the notebook's working directory. The `data/` folder is git-ignored; delete that subfolder to start from scratch.